# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhv-codes/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook converts the Week 6 client-holdout ranking into a small editorial review queue. It is decision support for a human editor, not a publishing system. The target is a decline of more than 20% in the later 30-day window versus the preceding 30 days. Features stop at the cutoff. The test labels are used only to report retrospective validation metrics; they are not exported in the queue.

The only model inputs are prior-window impressions, prior-window clicks, and content age at the cutoff. Client and content IDs are only used for grouping and stable tie-breaking. The data is the anonymized starter snapshot, not live client data.

## 1. Ranked actions + reason codes

The queue ranks eligible held-out pages by the client-holdout logistic regression risk score from the Week 6 audit. A tie is broken by prior-window impressions, then the pseudonymous content ID. The reason code describes model risk and the size of prior exposure; it is not a claim about why traffic changed. The exposure-at-stake field is risk probability multiplied by prior impressions, a sorting aid only. It is not recoverable traffic, revenue, or causal value.

The available validation shows that some pages meet the next-window decline label while others do not; the holdout model selected 49 of 50 qualifying declines in its top 50, compared with 46 of 50 for the prior-impressions rule. This is one snapshot and seven held-out clients, with a 91.1% decline base rate. It supports a review queue pilot, not a claim that refreshing content prevents decline. Age and refresh timing can help frame questions, but this observational snapshot cannot estimate the effect of refresh. The refresh signal is therefore a prompt to inspect source freshness and page history, not a refresh mandate.

In [2]:
from pathlib import Path
from hashlib import sha256
from urllib.request import urlopen
import json
import numpy as np
import pandas as pd
import sklearn

SEED = 42
RELATIVE_INPUT = Path("data/raw/content_refresh_anonymized.csv")
EXPECTED_SHA256 = "c43bdac4eccfa17fcd8a33974fe36f2c998c03a3ae3af8d80cf712abab5d6396"
DATA_COMMIT = "48d0dac47169e9f14b106037938d414b16529aaf"
here = Path.cwd()
repo_root = next((p for p in (here, *here.parents) if (p / RELATIVE_INPUT).is_file()), None)
if repo_root is None:
    # Colab fallback: fetch the exact public starter-data version used in the Week 6 audit.
    repo_root = here
    data_url = ("https://raw.githubusercontent.com/mhv-codes/flyrank-ml-internship/"
                f"{DATA_COMMIT}/{RELATIVE_INPUT.as_posix()}")
    with urlopen(data_url, timeout=60) as response:
        starter_csv = response.read()
    if sha256(starter_csv).hexdigest() != EXPECTED_SHA256:
        raise ValueError("Downloaded starter data does not match the verified version.")
    data_path = repo_root / RELATIVE_INPUT
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(starter_csv)
else:
    data_path = repo_root / RELATIVE_INPUT
    if sha256(data_path.read_bytes()).hexdigest() != EXPECTED_SHA256:
        raise ValueError("Local starter data does not match the verified Week 6 version.")

from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

keys = ["client_id", "content_id"]
feature_columns = ["impressions_prev_30d", "clicks_prev_30d", "age_at_cutoff"]
columns = keys + ["content_age_days", "days_since_last_update", "content_type",
                  "impressions_prev_30d", "clicks_prev_30d", "impressions_last_30d"]
data = pd.read_csv(data_path, usecols=columns)
assert len(data) == 30_000 and not data.duplicated(keys).any()
data["age_at_cutoff"] = data["content_age_days"] - 30
eligible = data.loc[data.age_at_cutoff.ge(30) & data.impressions_prev_30d.ge(500)].copy()
assert eligible[feature_columns + ["impressions_last_30d"]].notna().all().all()
assert eligible[feature_columns].ge(0).all().all()
eligible["decline"] = (5 * eligible.impressions_last_30d < 4 * eligible.impressions_prev_30d).astype(int)
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED)
                           .split(eligible, groups=eligible.client_id))
train, test = eligible.iloc[train_idx].copy(), eligible.iloc[test_idx].copy()
estimator = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(),
                          LogisticRegression(max_iter=1000, random_state=SEED))
model = clone(estimator).fit(train[feature_columns], train.decline)
test["risk_probability"] = model.predict_proba(test[feature_columns])[:, 1]
test["exposure_at_stake_proxy"] = test.risk_probability * test.impressions_prev_30d

def precision_at(frame, score, k):
    ranked = frame.sort_values([score, *keys], ascending=[False, True, True])
    return float(ranked.head(k).decline.mean())

baseline = test.assign(baseline_score=test.impressions_prev_30d)
metrics = {
    "dataset_sha256": EXPECTED_SHA256,
    "data_commit": DATA_COMMIT,
    "seed": SEED,
    "candidate_rows": int(len(eligible)),
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "train_clients": int(train.client_id.nunique()),
    "heldout_clients": int(test.client_id.nunique()),
    "client_overlap": int(len(set(train.client_id) & set(test.client_id))),
    "heldout_decline_base_rate": float(test.decline.mean()),
    "features": feature_columns,
    "label_definition": "impressions_last_30d < 0.8 * impressions_prev_30d",
    "week4_exposure_rule": {
        "precision_at_10": precision_at(baseline, "baseline_score", 10),
        "precision_at_50": precision_at(baseline, "baseline_score", 50),
        "average_precision": float(average_precision_score(test.decline, baseline.baseline_score)),
        "roc_auc": float(roc_auc_score(test.decline, baseline.baseline_score)),
    },
    "logistic_regression": {
        "precision_at_10": precision_at(test, "risk_probability", 10),
        "precision_at_50": precision_at(test, "risk_probability", 50),
        "average_precision": float(average_precision_score(test.decline, test.risk_probability)),
        "roc_auc": float(roc_auc_score(test.decline, test.risk_probability)),
    },
    "software": {"pandas": pd.__version__, "scikit_learn": sklearn.__version__},
}
print(f"Snapshot rows: {len(data):,}; eligible: {len(eligible):,}; client-holdout rows: {len(test):,} across {test.client_id.nunique()} clients.")
print(f"Train/test overlap: {metrics['client_overlap']} clients; held-out decline base rate: {metrics['heldout_decline_base_rate']:.3f}.")
print(pd.DataFrame([
    {"method":"Prior-impressions rule", **metrics["week4_exposure_rule"]},
    {"method":"Logistic regression", **metrics["logistic_regression"]},
]).set_index("method").round(4).to_string())
assert metrics["client_overlap"] == 0 and metrics["heldout_clients"] == 7
assert set(feature_columns).isdisjoint({"decline", "trend_pct", "trend_direction", "is_declining_label"})

Snapshot rows: 30,000; eligible: 11,119; client-holdout rows: 783 across 7 clients.
Train/test overlap: 0 clients; held-out decline base rate: 0.911.
                        precision_at_10  precision_at_50  average_precision  roc_auc
method                                                                              
Prior-impressions rule              0.8             0.92             0.9309   0.6205
Logistic regression                 1.0             0.98             0.9483   0.6463


## 2. Intended use and limits

An editor may use this list to decide which pages to inspect first when review time is limited. The score orders pages by model-estimated risk on the held-out clients; prior impressions provide context for potential exposure. Content archetype determines the first questions to ask, not the final edit.

This is a single anonymized snapshot with a 30-day outcome window. The held-out split contains seven clients and a high decline base rate, so its precision values are not portable guarantees. The queue is not causal, does not estimate traffic lift, and has no keyword-level intent evidence. It should not be used to promise ROI or automatically trigger a content refresh. A future time-based evaluation is needed before operational use.

In [3]:
archetype_actions = {
    "keyword article": ("KEYWORD_ARTICLE", "Check query intent, search demand, source freshness, and title-to-page alignment before proposing an edit."),
    "comparison article": ("COMPARISON_ARTICLE", "Recheck comparison criteria, product versions, dates, and evidence before changing any claim."),
    "feedly article": ("FEEDLY_ARTICLE", "Check whether the topic is time-sensitive and still useful; do not rewrite or archive without an editor's relevance check."),
}

exposure_threshold = eligible.impressions_prev_30d.quantile(.75)
def classify_reason(row):
    high_exposure = row.impressions_prev_30d >= exposure_threshold
    if row.risk_probability >= .70 and high_exposure:
        return "HIGH_RISK_HIGH_EXPOSURE"
    if row.risk_probability >= .70:
        return "HIGH_RISK"
    if row.risk_probability >= .50:
        return "ELEVATED_RISK"
    return "LOWER_RISK_MONITOR"

queue = test.copy()
queue["reason_code"] = queue.apply(classify_reason, axis=1)
queue["review_priority"] = np.select(
    [queue.risk_probability >= .70, queue.risk_probability >= .50],
    ["REVIEW_FIRST", "REVIEW_NEXT"], default="MONITOR")
queue["archetype"] = queue.content_type.map(lambda x: archetype_actions.get(x, ("UNKNOWN", "Manually classify before review."))[0])
queue["recommended_review"] = queue.content_type.map(lambda x: archetype_actions.get(x, ("UNKNOWN", "Manually classify before review."))[1])
queue["reason"] = queue.apply(
    lambda r: f"Estimated risk {r.risk_probability:.1%}; {int(r.impressions_prev_30d):,} impressions in the prior window.", axis=1)
queue = queue.sort_values(["risk_probability", "impressions_prev_30d", "content_id"],
                          ascending=[False, False, True]).reset_index(drop=True)
queue.insert(0, "review_rank", np.arange(1, len(queue) + 1))
queue["estimated_review_minutes"] = 15  # Planning assumption; replace with observed reviewer time.
export_columns = ["review_rank", "content_id", "review_priority", "reason_code", "archetype",
                  "risk_probability", "exposure_at_stake_proxy", "impressions_prev_30d",
                  "clicks_prev_30d", "age_at_cutoff", "days_since_last_update", "reason",
                  "recommended_review", "estimated_review_minutes"]
queue_export = queue[export_columns].copy()
print(f"{len(queue_export):,} held-out candidate rows ranked; top-10 review time at 15 min each: {10*15} minutes (planning assumption).")
print(queue_export.head(10).round({"risk_probability":3,"exposure_at_stake_proxy":1}).to_string(index=False))

783 held-out candidate rows ranked; top-10 review time at 15 min each: 150 minutes (planning assumption).
 review_rank           content_id review_priority             reason_code       archetype  risk_probability  exposure_at_stake_proxy  impressions_prev_30d  clicks_prev_30d  age_at_cutoff  days_since_last_update                                                        reason                                                                                         recommended_review  estimated_review_minutes
           1 content_88aa778b4b07    REVIEW_FIRST HIGH_RISK_HIGH_EXPOSURE KEYWORD_ARTICLE             0.886                  10925.9                 12331                0            102                      20 Estimated risk 88.6%; 12,331 impressions in the prior window. Check query intent, search demand, source freshness, and title-to-page alignment before proposing an edit.                        15
           2 content_16d67f7f1084    REVIEW_FIRST HIGH_RISK_HIGH_EXPOSURE KEYWORD_

## 3. Human review + the no-go list

Before action, a person must inspect the live page and the source material, confirm the page is still accurate, check search intent and current demand, distinguish tracking changes from real declines, and confirm that the suggested edit is appropriate for the reader. The editor records the decision and why. An observed fall in the later window is a flag to investigate, not proof that an edit will reverse it.

Never automate publishing, rewriting factual or comparative claims, changing canonical or indexing settings, deleting or merging a page, contacting a client, or allocating spend. Do not use this score to judge an author or make a client commitment. The model has no live URL, content text, query context, business priority, or legal review.

In [4]:
no_go_actions = ["publish_or_rewrite", "change_indexing_or_canonical", "delete_or_merge_page",
                 "contact_client", "allocate_budget", "make_revenue_or_traffic_guarantee"]
assert len(no_go_actions) == 6
print("Human-review gate: required before any edit. No-go actions:")
print("\n".join(f"- {x}" for x in no_go_actions))

Human-review gate: required before any edit. No-go actions:
- publish_or_rewrite
- change_indexing_or_canonical
- delete_or_merge_page
- contact_client
- allocate_budget
- make_revenue_or_traffic_guarantee


## 4. Monitoring / retrain triggers

For a future run, wait until each outcome window is complete, then compare a time-forward evaluation with the same eligibility rule and exposure baseline. Track queue size, client coverage, score calibration, precision at the review budget, and the rate of missing inputs. Review the model if held-out precision@50 falls by more than 10 percentage points from this reference run, average precision falls by more than 0.05, or the eligible queue size shifts by more than 25%. These are review thresholds, not automatic retraining commands.

Revisit the feature and label contract before retraining. Retrain only after a human confirms the data is comparable and the error analysis identifies a fixable problem. Preserve the old version and compare both on the same later window.

In [5]:
monitoring_triggers = {
    "precision_at_50_drop_pp": 10,
    "average_precision_drop": 0.05,
    "eligible_queue_volume_change_pct": 25,
    "action": "human review of data quality and time-forward performance; do not auto-retrain",
}
print(json.dumps(monitoring_triggers, indent=2))

{
  "precision_at_50_drop_pp": 10,
  "average_precision_drop": 0.05,
  "eligible_queue_volume_change_pct": 25,
  "action": "human review of data quality and time-forward performance; do not auto-retrain"
}


## 5. Exports for the paper

The CSV contains only the ranked review fields. It omits client IDs, page URLs, and outcome labels. Per repository policy, the queue CSV is regenerated by this notebook and stays out of Git. The metrics JSON is the committed receipt. No figure is needed for this playbook.

In [6]:
output_dir = repo_root / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
queue_path = output_dir / "content_action_queue.csv"
metrics_path = output_dir / "w07_action_playbook_metrics.json"
queue_export.to_csv(queue_path, index=False)
metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"Wrote queue: {queue_path.relative_to(repo_root)} ({queue_path.stat().st_size:,} bytes; ignored by Git)")
print(f"Wrote metrics receipt: {metrics_path.relative_to(repo_root)}")
assert queue_path.exists() and metrics_path.exists()
roundtrip = pd.read_csv(queue_path)
assert len(roundtrip) == len(test) and not {"client_id", "decline", "impressions_last_30d"}.intersection(roundtrip.columns)
print(f"Export check passed: {len(roundtrip):,} rows; no client IDs or outcome labels exported.")

Wrote queue: work/outputs/content_action_queue.csv (229,875 bytes; ignored by Git)
Wrote metrics receipt: work/outputs/w07_action_playbook_metrics.json
Export check passed: 783 rows; no client IDs or outcome labels exported.


## 6. Self-check

- [x] Ranked review actions have reason codes and an archetype-to-action mapping.
- [x] Intended use, observational decay/refresh limits, human-review rules, no-go cases, and cost/value assumptions are stated.
- [x] Monitoring and retraining triggers are review thresholds, not automatic production actions.
- [x] Inputs use only prior-window measures and cutoff age; the later-window label is reserved for evaluation.
- [x] The notebook writes the ranked CSV and a committed metrics JSON receipt. The CSV stays out of Git by design.
- [x] This is a practical, offline decision-support example, not a production system.